# Module 6 • Transformers

# Lesson 34 • Transformer Encoder–Decoder Models for Conditional Generation

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Execution target:** CPU

## Scope
This lesson develops Transformer encoder–decoder models for conditional generation. It covers source self-attention, causal target self-attention, cross-attention, source and target masks, shifted targets, teacher-forced training, greedy decoding, beam-search foundations, sequence metrics, error analysis, and Arabic/multilingual considerations.

## Learning Objectives

By the end of this lesson, you should be able to explain the three attention pathways in an encoder–decoder Transformer, construct all required masks, train a compact translation model, decode autoregressively, compare greedy and beam search, and evaluate generation quality.

## Table of Contents

1. Conditional generation  
2. Encoder–decoder Transformer architecture  
3. Self-attention and cross-attention  
4. Source and target masks  
5. Shifted targets and masked loss  
6. Translation dataset and vocabularies  
7. Transformer implementation  
8. Training and validation  
9. Greedy decoding  
10. Beam-search foundations  
11. Evaluation and error analysis  
12. Computational and multilingual considerations  
13. Knowledge check and exercises  
14. Summary and next lesson

# 1. Conditional Generation
Conditional generation maps an input sequence to an output sequence. Machine translation, summarization, correction, and data-to-text generation are canonical examples.

In [ ]:
import copy
import math
import random
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Dataset

pd.DataFrame([
    ("Translation", "source sentence", "target sentence"),
    ("Summarization", "document", "summary"),
    ("Correction", "noisy text", "corrected text"),
], columns=["Task","Condition","Generated output"])

# 2. Encoder–Decoder Transformer Architecture

The encoder converts the source into contextual memory. The decoder produces the target left-to-right while consulting both its target prefix and the encoder memory.

In [ ]:
pd.DataFrame([
    ("Encoder self-attention", "source", "source"),
    ("Decoder self-attention", "target prefix", "target prefix"),
    ("Cross-attention", "decoder state", "encoder memory"),
], columns=["Attention pathway","Queries","Keys and values"])

# 3. Self-Attention and Cross-Attention

Encoder self-attention is bidirectional over valid source tokens. Decoder self-attention is causal. Cross-attention connects each target position to all valid source positions.

# 4. Source and Target Masks

A source padding mask blocks source PAD positions. A target padding mask blocks target PAD positions. A target causal mask prevents access to future target tokens.

In [ ]:
def causal_mask(length: int) -> torch.Tensor:
    return torch.triu(torch.ones(length, length, dtype=torch.bool), diagonal=1)

pd.DataFrame(causal_mask(6).int().numpy())

In [ ]:
pd.DataFrame([
    ("Source padding mask", "(B, S)", "source PAD keys"),
    ("Target padding mask", "(B, T)", "target PAD keys"),
    ("Target causal mask", "(T, T)", "future target keys"),
    ("Memory padding mask", "(B, S)", "source PAD during cross-attention"),
], columns=["Mask","Shape","Blocks"])

# 5. Shifted Targets and Masked Loss

For `<BOS> ouvre la porte rouge <EOS>`, the decoder input ends before EOS and the expected target starts after BOS.

In [ ]:
sequence=["<BOS>","ouvre","la","porte","rouge","<EOS>"]
pd.DataFrame({"decoder_input":sequence[:-1],"expected_target":sequence[1:]})

# 6. Toy Translation Dataset

The controlled English-to-French task includes lexical translation, article selection, adjective agreement, and adjective reordering.

In [ ]:
verbs={"open":"ouvre","close":"ferme","find":"trouve","take":"prends","move":"deplace"}
nouns={
    "door":("la","porte","f"),"window":("la","fenetre","f"),
    "box":("la","boite","f"),"book":("le","livre","m"),"key":("la","cle","f")
}
modifiers={
    "red":("after","rouge","rouge"),"blue":("after","bleu","bleue"),
    "green":("after","vert","verte"),"yellow":("after","jaune","jaune"),
    "small":("before","petit","petite"),"big":("before","grand","grande")
}

def build_pair(v,n,m):
    article,noun_fr,gender=nouns[n]
    position,male,female=modifiers[m]
    adj=male if gender=="m" else female
    src=f"{v} the {m} {n}"
    tgt=(f"{verbs[v]} {article} {adj} {noun_fr}" if position=="before" else f"{verbs[v]} {article} {noun_fr} {adj}")
    return src,tgt

records=[]
for v in verbs:
    for n in nouns:
        for m in modifiers:
            s,t=build_pair(v,n,m)
            records.append((s,t,v,n,m))
dataset=pd.DataFrame(records,columns=["source","target","verb","noun","modifier"])
print("Pairs:",len(dataset))
dataset.sample(8,random_state=42)[["source","target"]].reset_index(drop=True)

# 7. Train, Validation, and Test Splits

In [ ]:
train_frame,test_frame=train_test_split(dataset,test_size=0.15,random_state=42)
train_frame,validation_frame=train_test_split(train_frame,test_size=0.1765,random_state=42)
train_frame=train_frame.reset_index(drop=True)
validation_frame=validation_frame.reset_index(drop=True)
test_frame=test_frame.reset_index(drop=True)
pd.Series({"training":len(train_frame),"validation":len(validation_frame),"test":len(test_frame)})

# 8. Tokenization and Vocabularies

In [ ]:
TOKEN_PATTERN=re.compile(r"\b\w+(?:[-']\w+)*\b",flags=re.UNICODE)
def tokenize(text): return TOKEN_PATTERN.findall(text.lower())

PAD,UNK,BOS,EOS="<PAD>","<UNK>","<BOS>","<EOS>"
class Vocabulary:
    def __init__(self,texts,include_bos=False):
        counts=Counter(tok for text in texts for tok in tokenize(text))
        specials=[PAD,UNK]+([BOS] if include_bos else [])+[EOS]
        self.itos=specials+sorted(counts)
        self.stoi={t:i for i,t in enumerate(self.itos)}
        self.pad_id=self.stoi[PAD]; self.unk_id=self.stoi[UNK]; self.eos_id=self.stoi[EOS]
        self.bos_id=self.stoi[BOS] if include_bos else None
    def __len__(self): return len(self.itos)
    def encode(self,text,add_bos=False):
        ids=[self.stoi.get(t,self.unk_id) for t in tokenize(text)]
        return (([self.bos_id] if add_bos else [])+ids+[self.eos_id])
    def decode(self,ids):
        out=[]
        for i in ids:
            tok=self.itos[int(i)]
            if tok==EOS: break
            if tok not in {PAD,BOS}: out.append(tok)
        return out

source_vocab=Vocabulary(train_frame.source)
target_vocab=Vocabulary(train_frame.target,include_bos=True)
print(len(source_vocab),len(target_vocab))

# 9. Dataset and Dynamic Padding

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self,frame): self.frame=frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self,index):
        r=self.frame.iloc[index]
        return {
            "source_ids":torch.tensor(source_vocab.encode(r.source),dtype=torch.long),
            "target_ids":torch.tensor(target_vocab.encode(r.target,add_bos=True),dtype=torch.long),
            "source_text":r.source,"target_text":r.target,
        }

def collate_batch(batch):
    ms=max(len(x["source_ids"]) for x in batch); mt=max(len(x["target_ids"]) for x in batch)
    src=torch.full((len(batch),ms),source_vocab.pad_id,dtype=torch.long)
    tgt=torch.full((len(batch),mt),target_vocab.pad_id,dtype=torch.long)
    for i,x in enumerate(batch):
        src[i,:len(x["source_ids"])]=x["source_ids"]
        tgt[i,:len(x["target_ids"])]=x["target_ids"]
    return {"source_ids":src,"target_ids":tgt,"source_padding_mask":src.eq(source_vocab.pad_id),
            "source_texts":[x["source_text"] for x in batch],"target_texts":[x["target_text"] for x in batch]}

train_loader=DataLoader(TranslationDataset(train_frame),batch_size=16,shuffle=True,collate_fn=collate_batch,generator=torch.Generator().manual_seed(42))
validation_loader=DataLoader(TranslationDataset(validation_frame),batch_size=16,shuffle=False,collate_fn=collate_batch)
test_loader=DataLoader(TranslationDataset(test_frame),batch_size=16,shuffle=False,collate_fn=collate_batch)
sample_batch=next(iter(train_loader))
print(sample_batch["source_ids"].shape,sample_batch["target_ids"].shape)

# 10. Positional Encoding

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self,d_model,max_len=64):
        super().__init__()
        pe=torch.zeros(max_len,d_model)
        pos=torch.arange(max_len,dtype=torch.float32).unsqueeze(1)
        rates=torch.exp(torch.arange(0,d_model,2,dtype=torch.float32)*(-math.log(10000.0)/d_model))
        pe[:,0::2]=torch.sin(pos*rates); pe[:,1::2]=torch.cos(pos*rates)
        self.register_buffer("pe",pe.unsqueeze(0))
    def forward(self,x): return x+self.pe[:,:x.size(1)]

# 11. Transformer Encoder–Decoder Implementation

In [ ]:
class TransformerSeq2Seq(nn.Module):
    def __init__(self,src_size,tgt_size,d_model=32,nhead=4,layers=2,ff=64,dropout=0.1):
        super().__init__(); self.d_model=d_model
        self.src_emb=nn.Embedding(src_size,d_model,padding_idx=source_vocab.pad_id)
        self.tgt_emb=nn.Embedding(tgt_size,d_model,padding_idx=target_vocab.pad_id)
        self.position=SinusoidalPositionalEncoding(d_model)
        self.transformer=nn.Transformer(d_model=d_model,nhead=nhead,num_encoder_layers=layers,num_decoder_layers=layers,
                                        dim_feedforward=ff,dropout=dropout,activation="gelu",batch_first=True,norm_first=True)
        self.output=nn.Linear(d_model,tgt_size)
    def target_mask(self,length,device): return torch.triu(torch.ones(length,length,dtype=torch.bool,device=device),diagonal=1)
    def encode(self,src,src_pad):
        x=self.position(self.src_emb(src)*math.sqrt(self.d_model))
        return self.transformer.encoder(x,src_key_padding_mask=src_pad)
    def decode(self,tgt,memory,tgt_pad,src_pad):
        y=self.position(self.tgt_emb(tgt)*math.sqrt(self.d_model))
        return self.transformer.decoder(y,memory,tgt_mask=self.target_mask(tgt.size(1),tgt.device),
                                        tgt_key_padding_mask=tgt_pad,memory_key_padding_mask=src_pad)
    def forward(self,src,tgt_in,src_pad,tgt_pad):
        memory=self.encode(src,src_pad); decoded=self.decode(tgt_in,memory,tgt_pad,src_pad)
        return {"logits":self.output(decoded),"memory":memory,"decoded":decoded}

DEVICE=torch.device("cpu")
torch.manual_seed(42)
model=TransformerSeq2Seq(len(source_vocab),len(target_vocab)).to(DEVICE)
print("Parameters:",sum(p.numel() for p in model.parameters()))

# 12. Shape and Mask Inspection

In [ ]:
src=sample_batch["source_ids"].to(DEVICE); tgt=sample_batch["target_ids"].to(DEVICE)
tgt_in=tgt[:,:-1]; tgt_pad=tgt_in.eq(target_vocab.pad_id)
with torch.no_grad(): out=model(src,tgt_in,sample_batch["source_padding_mask"].to(DEVICE),tgt_pad)
print(out["memory"].shape,out["decoded"].shape,out["logits"].shape)
pd.DataFrame(model.target_mask(tgt_in.size(1),DEVICE).int().cpu().numpy())

# 13. Training Utilities

In [ ]:
loss_fn=nn.CrossEntropyLoss(ignore_index=target_vocab.pad_id)
def seq_loss(logits,targets): return loss_fn(logits.reshape(-1,logits.size(-1)),targets.reshape(-1))
def set_seed(seed=42): random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def evaluate_loss(model,loader):
    model.eval(); vals=[]
    with torch.no_grad():
        for b in loader:
            src=b["source_ids"].to(DEVICE); tgt=b["target_ids"].to(DEVICE)
            ti,expected=tgt[:,:-1],tgt[:,1:]
            output=model(src,ti,b["source_padding_mask"].to(DEVICE),ti.eq(target_vocab.pad_id))
            vals.append(float(seq_loss(output["logits"],expected)))
    return float(np.mean(vals))

# 14. Training the Model

In [ ]:
def train_model(model,epochs=40,lr=0.003,patience=8):
    opt=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=1e-4)
    best=copy.deepcopy(model.state_dict()); best_val=float("inf"); wait=0; history=[]
    for epoch in range(epochs):
        model.train(); losses=[]; norms=[]
        for b in train_loader:
            src=b["source_ids"].to(DEVICE); tgt=b["target_ids"].to(DEVICE)
            ti,expected=tgt[:,:-1],tgt[:,1:]
            opt.zero_grad(); output=model(src,ti,b["source_padding_mask"].to(DEVICE),ti.eq(target_vocab.pad_id))
            loss=seq_loss(output["logits"],expected); loss.backward()
            norms.append(float(clip_grad_norm_(model.parameters(),5.0))); opt.step(); losses.append(float(loss))
        val=evaluate_loss(model,validation_loader)
        history.append({"epoch":epoch,"train_loss":float(np.mean(losses)),"validation_loss":val,
                        "validation_perplexity":math.exp(min(val,20.0)),"gradient_norm":float(np.mean(norms))})
        if val<best_val-1e-5: best_val=val; best=copy.deepcopy(model.state_dict()); wait=0
        else: wait+=1
        if wait>=patience: break
    model.load_state_dict(best); return model,pd.DataFrame(history)

set_seed(42)
trained_model,history=train_model(model)
print("Epochs:",len(history),"best val:",round(history.validation_loss.min(),4))

# 15. Learning Curves

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.epoch,history.train_loss,label="Training loss")
plt.plot(history.epoch,history.validation_loss,label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Transformer Translation Learning Curves"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.epoch,history.validation_perplexity)
plt.xlabel("Epoch"); plt.ylabel("Perplexity"); plt.title("Validation Perplexity"); plt.tight_layout(); plt.show()

# 16. Greedy Decoding

In [ ]:
def greedy_decode(model,source_text,max_new_tokens=8):
    model.eval(); src=torch.tensor([source_vocab.encode(source_text)],dtype=torch.long,device=DEVICE); src_pad=src.eq(source_vocab.pad_id)
    with torch.no_grad():
        memory=model.encode(src,src_pad); generated=[target_vocab.bos_id]; probs=[]
        for _ in range(max_new_tokens):
            ti=torch.tensor([generated],dtype=torch.long,device=DEVICE)
            decoded=model.decode(ti,memory,ti.eq(target_vocab.pad_id),src_pad)
            logits=model.output(decoded[:,-1]); p=torch.softmax(logits,dim=1); nxt=int(p.argmax(1).item())
            generated.append(nxt); probs.append(float(p[0,nxt]))
            if nxt==target_vocab.eos_id: break
    return {"translation":" ".join(target_vocab.decode(generated)),"ids":generated,"probabilities":probs}

greedy_decode(trained_model,"open the red door")

# 17. Qualitative Translation

In [ ]:
rows=[]
for r in test_frame.head(10).itertuples(index=False):
    result=greedy_decode(trained_model,r.source)
    rows.append({"source":r.source,"reference":r.target,"prediction":result["translation"],
                 "confidence":float(np.mean(result["probabilities"]))})
pd.DataFrame(rows)

# 18. Beam-Search Foundations

Beam search keeps several partial hypotheses and ranks them by accumulated log probability, often with length normalization.

In [ ]:
def normalized_score(log_probability,length,alpha=0.7):
    return log_probability/(max(length,1)**alpha)

pd.DataFrame([(-2.0,4,normalized_score(-2.0,4)),(-2.5,6,normalized_score(-2.5,6))],
             columns=["log_probability","length","normalized_score"])

In [ ]:
def beam_decode(model,source_text,beam_width=3,max_new_tokens=8,alpha=0.7):
    model.eval(); src=torch.tensor([source_vocab.encode(source_text)],dtype=torch.long,device=DEVICE); src_pad=src.eq(source_vocab.pad_id)
    with torch.no_grad():
        memory=model.encode(src,src_pad); beams=[([target_vocab.bos_id],0.0,False)]
        for _ in range(max_new_tokens):
            candidates=[]
            for ids,score,finished in beams:
                if finished: candidates.append((ids,score,True)); continue
                ti=torch.tensor([ids],dtype=torch.long,device=DEVICE)
                decoded=model.decode(ti,memory,ti.eq(target_vocab.pad_id),src_pad)
                logp=torch.log_softmax(model.output(decoded[:,-1]),dim=1)[0]
                values,indices=torch.topk(logp,k=min(beam_width,logp.numel()))
                for v,i in zip(values.tolist(),indices.tolist()):
                    new_ids=ids+[int(i)]; candidates.append((new_ids,score+float(v),int(i)==target_vocab.eos_id))
            beams=sorted(candidates,key=lambda x:normalized_score(x[1],len(x[0]),alpha),reverse=True)[:beam_width]
            if all(b[2] for b in beams): break
    best=beams[0]
    return " ".join(target_vocab.decode(best[0]))

beam_decode(trained_model,"find the small key")

# 19. Translation Evaluation

In [ ]:
def positional_accuracy(reference,prediction):
    r,p=tokenize(reference),tokenize(prediction); denom=max(len(r),len(p),1)
    return sum(a==b for a,b in zip(r,p))/denom

def ngrams(tokens,n): return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))
def bleu_like(reference,prediction,max_order=4):
    r,p=tokenize(reference),tokenize(prediction)
    if not p: return 0.0
    precisions=[]
    for n in range(1,max_order+1):
        pn,rn=ngrams(p,n),ngrams(r,n); clipped=sum(min(c,rn[g]) for g,c in pn.items()); total=sum(pn.values())
        precisions.append((clipped+1)/(total+1))
    bp=1.0 if len(p)>len(r) else math.exp(1-len(r)/max(len(p),1))
    return bp*math.exp(sum(math.log(max(x,1e-12)) for x in precisions)/max_order)

def evaluate(frame):
    rows=[]
    for r in frame.itertuples(index=False):
        result=greedy_decode(trained_model,r.source); pred=result["translation"]
        rows.append({"source":r.source,"reference":r.target,"prediction":pred,"exact_match":pred==r.target,
                     "token_accuracy":positional_accuracy(r.target,pred),"bleu_like":bleu_like(r.target,pred),
                     "confidence":float(np.mean(result["probabilities"]))})
    return pd.DataFrame(rows)

test_results=evaluate(test_frame)
pd.Series({"exact_match":test_results.exact_match.mean(),"token_accuracy":test_results.token_accuracy.mean(),"bleu_like":test_results.bleu_like.mean()}).round(3)

# 20. Error Analysis

In [ ]:
def error_type(row):
    r,p=tokenize(row.reference),tokenize(row.prediction)
    if row.exact_match: return "correct"
    if len(p)<len(r): return "too short"
    if len(p)>len(r): return "too long"
    if set(p)==set(r): return "word order"
    return "lexical or agreement"

test_results["error_type"]=test_results.apply(error_type,axis=1)
test_results.sort_values(["exact_match","bleu_like","confidence"],ascending=[True,True,False]).reset_index(drop=True)

In [ ]:
test_results.error_type.value_counts()

# 21. Cross-Attention Interpretation
Cross-attention can reveal model focus, but attention weights are not guaranteed explanations. Reliable interpretation requires comparing heads, layers, seeds, perturbations, and output behavior.

# 22. Teacher Forcing and Exposure Bias
During training, decoder prefixes are gold targets. During inference, prefixes are model predictions. This mismatch can amplify early errors.

In [ ]:
pd.DataFrame([
    ("Training","gold target prefix","clean history"),
    ("Inference","predicted prefix","error accumulation"),
],columns=["Phase","Prefix","Consequence"])

# 23. Computational Cost

In [ ]:
pd.DataFrame([
    ("Encoder self-attention","source_length²"),
    ("Decoder self-attention","target_length²"),
    ("Cross-attention","source_length × target_length"),
    ("Beam search","beam_width × decoding cost"),
],columns=["Component","Approximate scaling"])

# 24. Common Failure Modes

Typical failures include target leakage, incorrect mask polarity, PAD contamination, premature EOS, repetition, lexical substitutions, agreement errors, and poor length control.

In [ ]:
pd.DataFrame([
    ("Target leakage","verify causal target mask"),
    ("PAD contamination","use padding masks and ignore_index"),
    ("Premature EOS","improve supervision and length handling"),
    ("Repetition","beam constraints or coverage methods"),
    ("Overfitting","dropout, early stopping, more data"),
],columns=["Failure","Response"])

# 25. Arabic and Multilingual Considerations

Arabic conditional generation must account for clitic attachment, rich morphology, optional tashkeel, agreement, word-order differences, MSA and dialects, code-switching, and subword fragmentation.

In [ ]:
pd.DataFrame([
    ("وَسَيَكْتُبُونَهَا","وَ + سَ + يَكْتُبُونَ + هَا"),
    ("بِالْمَدْرَسَةِ","بِ + الْمَدْرَسَةِ"),
    ("كِتَابُهُمَا","كِتَابُ + هُمَا"),
],columns=["Fully vocalized form","Illustrative segmentation"])

Source and target tokenization may use separate vocabularies or a shared multilingual subword vocabulary. For fully vocalized Arabic tasks, tashkeel must remain in both source and target sequences whenever it is part of the task definition.

# 26. Reproducibility and Reporting

In [ ]:
import platform
pd.Series({
    "sentence_pairs":len(dataset),"training_pairs":len(train_frame),"validation_pairs":len(validation_frame),"test_pairs":len(test_frame),
    "source_vocabulary":len(source_vocab),"target_vocabulary":len(target_vocab),"model_dimension":32,"attention_heads":4,
    "encoder_layers":2,"decoder_layers":2,"device":str(DEVICE),"random_seed":42,
    "python_version":platform.python_version(),"torch_version":torch.__version__,
},name="Transformer translation experiment")

# 27. Knowledge Check

1. What is conditional generation?  
2. Why is decoder self-attention causal?  
3. What does cross-attention connect?  
4. Which masks are required?  
5. Why are targets shifted?  
6. What is target leakage?  
7. Why must PAD targets be ignored?  
8. How does greedy decoding work?  
9. What does beam search retain?  
10. Why use length normalization?  
11. What is exposure bias?  
12. How do Arabic morphology and tashkeel affect generation?

# 28. Exercises

1. Implement custom encoder and decoder layers that return attention weights.  
2. Compare one and two encoder–decoder layers.  
3. Compare two and four heads.  
4. Implement beam search with a coverage penalty.  
5. Add label smoothing.  
6. Add a warmup learning-rate schedule.  
7. Add established BLEU and chrF implementations.  
8. Train with a shared vocabulary.  
9. Build a fully vocalized Arabic translation task.  
10. Compare recurrent and Transformer encoder–decoders under matched parameter budgets.

# 29. Summary and Next Lesson

This lesson developed a complete Transformer encoder–decoder workflow: source encoding, causal target decoding, cross-attention, masking, shifted training targets, greedy and beam decoding, sequence metrics, and error analysis.

## Next Lesson

**Lesson 35: Pretrained Transformer Models and Hugging Face Workflows** introduces tokenizers, checkpoints, pipelines, feature extraction, sequence classification, model selection, caching, and responsible use of pretrained models.

# References

- Vaswani et al., *Attention Is All You Need*.  
- Sutskever, Vinyals, and Le, sequence-to-sequence learning.  
- Bahdanau, Cho, and Bengio, neural machine translation with alignment.  
- Papineni et al., BLEU.  
- Jurafsky and Martin, *Speech and Language Processing*.